In [1]:
import os
import glob
import os
import pandas as pd
import numpy as np
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from tqdm import tqdm
import time
from scipy.spatial.distance import cdist
import scipy
from scanpy.tools._utils import get_init_pos_from_paga 

import rmm
import cupy
import cudf
import cupy as cp
import cvxpy as cpv
from cuml.metrics import pairwise_distances
from cuml.metrics import nan_euclidean_distances
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
%%time
fpath = "../../resources/gene_names.tsv.gz"
gdf = pd.read_csv(fpath, sep='\t', low_memory=False)
print(f"{gdf.shape=}")

protein_coding = gdf[gdf['Gene type'] == 'protein_coding']['Gene name'].unique()
print(f"N protein coding: {protein_coding.shape=}")

gdf.shape=(73466, 8)
N protein coding: protein_coding.shape=(18149,)
CPU times: user 129 ms, sys: 19.9 ms, total: 149 ms
Wall time: 149 ms


# Define Datasets

In [3]:
datasets = {
    'Reference' :  {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hematokytos/sample_1_adata.h5ad",
        'annotation_column' : 'cell_type',
        'layer' : 'counts',
        'filter_column' : 'basename',
        'filter_out' : ['myeloid_cells', 'lymphoid_cells', 'innate_lymphoid_cells'],
    },
    'Ng 2024' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/ng_2024/iHSC.h5ad",
        'annotation_column' : 'celltype',
        'layer' : None,
        'filter_column' : None,
        'filter_out' : None,
    },
    'Gomes 2018' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/gomes_2018/gomes.h5ad",
        'annotation_column' : 'cell_type',
        'layer' : 'raw_counts',
        'filter_column' : None,
        'filter_out' : None,
    },
    'This Study' : {
        'file_path' : "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/three_libraries.h5ad",
        'annotation_column' : 'subgroup',
        'layer' : 'raw_counts',
        'filter_column' : None,
        'filter_out' : None,
    },
}

In [4]:
%%time
data = {}

for dataset, params in datasets.items():
    print(f"\n--- Loading dataset: {dataset} ---")
    t0 = time.time()

    fpath = params['file_path']
    column = params['annotation_column']
    layer = params['layer']
    filter_column = params['filter_column']
    filter_out = params['filter_out']

    print(f"\t - Reading file: {fpath}")
    bdata = sc.read_h5ad(fpath)
    bdata.var_names_make_unique()
    print(f"\t - Read complete. Shape: {bdata.shape}")
    print(f"\t - Obs columns: {', '.join(bdata.obs.columns)}")

    # Annotate cell type and filter
    bdata.obs['annotation'] = bdata.obs[column].copy()
    n_total = bdata.shape[0]
    bdata = bdata[bdata.obs['annotation'].notna(), :].copy()
    n_filtered = bdata.shape[0]
    print(f"\t - Cells before filtering: {n_total}, after filtering: {n_filtered}")

     # Select layer if specified
    if layer is not None:
        print(f"\t - Using layer: {layer}")
        bdata.X = bdata.layers[layer].copy()
    else:
        print("\t - Using .X as is (no layer specified)")

    
    print(f"\t - Filtering...")
    if filter_column is None:
        print(f"\t - No Defined filtering.")
    else:
        print(f"\t - Filtering {filter_out} from {filter_column}...")
        bdata = bdata[~bdata.obs[filter_column].isin(filter_out), :].copy()

    print("\t - Filtering non coding genes...")
    bdata = bdata[:, bdata.var_names.isin(protein_coding)].copy()
    
    bdata.X = bdata.X.astype('float32')
    rsc.get.anndata_to_GPU(bdata)
    rsc.pp.filter_genes(bdata, min_counts=10)
    
    print(f"\t - Final Shape: {bdata.shape}")

    n_genes = bdata.shape[1]
    n_cell_types = bdata.obs['annotation'].nunique()
    cell_types = sorted(bdata.obs['annotation'].unique())
    print(f"\t - Unique cell types ({n_cell_types}): {cell_types[:5]}{' ...' if n_cell_types > 5 else ''}")

    print(f"\t - Cleaning up metadata...")
    del bdata.obsm
    del bdata.obsp
    del bdata.layers
    del bdata.varm

    data[dataset] = bdata

    t1 = time.time()
    print(f"\t - Time to load and process: {(t1 - t0)/60:.2f} minutes")

print()
print(data.keys())
print('done.')


--- Loading dataset: Reference ---
	 - Reading file: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hematokytos/sample_1_adata.h5ad


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Read complete. Shape: (1125041, 52164)
	 - Obs columns: soma_joinid, dataset_id, assay, assay_ontology_term_id, cell_type, cell_type_ontology_term_id, development_stage, development_stage_ontology_term_id, disease, disease_ontology_term_id, donor_id, is_primary_data, observation_joinid, self_reported_ethnicity, self_reported_ethnicity_ontology_term_id, sex, sex_ontology_term_id, suspension_type, tissue, tissue_ontology_term_id, tissue_type, tissue_general, tissue_general_ontology_term_id, raw_sum, nnz, raw_mean_nnz, raw_variance_nnz, n_measured_vars, basename, dataset_id_int, n_counts, n_genes, _scvi_batch, _scvi_labels, leiden


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Cells before filtering: 1125041, after filtering: 1125041
	 - Using layer: counts
	 - Filtering...
	 - Filtering ['myeloid_cells', 'lymphoid_cells', 'innate_lymphoid_cells'] from basename...


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


	 - Filtering non coding genes...
filtered out 295 genes that are detected in less than 10 counts
	 - Final Shape: (709808, 17524)
	 - Unique cell types (72): ['CD34-positive, CD38-negative hematopoietic stem cell', 'adipocyte', 'alveolar type 1 fibroblast cell', 'alveolar type 2 fibroblast cell', 'basophil mast progenitor cell'] ...
	 - Cleaning up metadata...
	 - Time to load and process: 5.65 minutes

--- Loading dataset: Ng 2024 ---
	 - Reading file: /nfs/turbo/umms-indikar/shared/projects/HSC/data/datasets/ng_2024/iHSC.h5ad
	 - Read complete. Shape: (252607, 27946)
	 - Obs columns: orig.ident, nCount_RNA, nFeature_RNA, line.ident, percent.mt, percent.ribo, percent.mitoribo, S.Score, G2M.Score, Phase, integrated_snn_res.0.5, seurat_clusters, integrated_snn_res.1, integrated_snn_res.0.8, integrated_snn_res.0.3, integrated_snn_res.0.4, fract.ident, induction.ident, celltype, RETA
	 - Cells before filtering: 252607, after filtering: 252607
	 - Using .X as is (no layer specified)
	 - F

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


	 - Cells before filtering: 286, after filtering: 286
	 - Using layer: raw_counts
	 - Filtering...
	 - No Defined filtering.
	 - Filtering non coding genes...
filtered out 3550 genes that are detected in less than 10 counts
	 - Final Shape: (286, 13492)
	 - Unique cell types (5): ['DAY15', 'DAY2', 'DAY25', 'HDF', 'UCB']
	 - Cleaning up metadata...
	 - Time to load and process: 0.00 minutes

--- Loading dataset: This Study ---
	 - Reading file: /nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/three_libraries.h5ad
	 - Read complete. Shape: (21062, 25739)
	 - Obs columns: dataset, source, cluster, group, label, S_score, G2M_score, phase, n_genes_by_counts, log1p_n_genes_by_counts, total_counts, log1p_total_counts, pct_counts_in_top_50_genes, pct_counts_in_top_100_genes, pct_counts_in_top_200_genes, pct_counts_in_top_500_genes, total_counts_mito, log1p_total_counts_mito, pct_counts_mito, total_counts_ribo, log1p_total_counts_ribo, pct_counts_ribo, filter_pass

In [5]:
for k, v in data.items():
    print(k, v.shape)
    

Reference (709808, 17524)
Ng 2024 (252607, 15023)
Gomes 2018 (286, 13492)
This Study (21062, 14933)


In [6]:
for k, v in data.items():
    print('\n', k, list(v.obs.columns))


 Reference ['soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden', 'annotation']

 Ng 2024 ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'line.ident', 'percent.mt', 'percent.ribo', 'percent.mitoribo', 'S.Score', 'G2M.Score', 'Phase', 'integrated_snn_res.0.5', 'seurat_clusters', 'integrated_snn_res.1', 'integrated_snn_res.0.8', 'integrated_snn_res.0.3', 'integrated_snn_res.0.4', 'fract.ident', 'inducti

In [7]:
%%time
max_cells = 250000
keep_columns = [
    'annotation',
    'cell_type',
    'celltype', 
    'tissue',
    'development_stage',
    'basename',
    'source', 
    'dataset_id',
]

adatas = []

print(f"Sampling up to {max_cells} cells per dataset...\n")

for key, bdata in data.items():
    print(f"Processing dataset: {key}")
    obs = bdata.obs.copy()
    
    # Keep only selected columns
    kept = [col for col in keep_columns if col in obs.columns]
    missing = [col for col in keep_columns if col not in obs.columns]
    print(f"  Keeping columns: {kept}")
    if missing:
        print(f"  Warning: missing columns: {missing}")
    
    obs = obs[kept]
    bdata.obs = obs
    bdata.obs['data_key'] = key
    bdata.obs['data_source'] = key

    # Sample cells
    n_cells = min(bdata.n_obs, max_cells)
    print(f"  Sampling {n_cells} cells (of {bdata.n_obs})")
    bdata = sc.pp.sample(bdata, n=n_cells, replace=False, copy=True)
    
    adatas.append(bdata)

print("\nConcatenating all datasets with outer join on genes...")
adata = an.concat(
    adatas, 
    join='outer',
    label='data_source',
    index_unique=None,
)

print(f"Final AnnData object: {adata.shape[0]} cells × {adata.shape[1]} genes")

adata

Sampling up to 250000 cells per dataset...

Processing dataset: Reference
  Keeping columns: ['annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id']
  Sampling 250000 cells (of 709808)


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Processing dataset: Ng 2024
  Keeping columns: ['annotation', 'celltype']
  Sampling 250000 cells (of 252607)
Processing dataset: Gomes 2018
  Keeping columns: ['annotation', 'cell_type']
  Sampling 286 cells (of 286)
Processing dataset: This Study
  Keeping columns: ['annotation', 'source']
  Sampling 21062 cells (of 21062)

Concatenating all datasets with outer join on genes...


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/cupyx/scipy/sparse/_compressed.py:548: SparseEfficiencyWarning: Changing the sparsity structure of a csc_matrix is expensive.
  warnings.warn('Changing the sparsity structure of a '


Final AnnData object: 521348 cells × 17601 genes
CPU times: user 1min 50s, sys: 9.98 s, total: 2min
Wall time: 2min 1s


/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/merge.py:1434: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 521348 × 17601
    obs: 'annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id', 'data_key', 'data_source', 'celltype', 'source'

# store 

In [8]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/single_cell_atlas.h5ad"
adata.write(outpath)
adata

CPU times: user 8.91 s, sys: 19.2 s, total: 28.1 s
Wall time: 58.9 s


AnnData object with n_obs × n_vars = 521348 × 17601
    obs: 'annotation', 'cell_type', 'tissue', 'development_stage', 'basename', 'dataset_id', 'data_key', 'data_source', 'celltype', 'source'